# CartSense Apriori Evaluation Pipeline

This notebook is the plain, reproducible reference for the CartSense Python implementation. It evaluates order-level basket recommendations on `public/dataset/modeldataset.csv`.

Apriori is an association-rule method, not a supervised classifier. Therefore the primary performance measures are recommendation metrics: hit rate, recall, mean reciprocal rank, rule coverage, recommendation coverage, and comparison with a popularity baseline. Rule confidence is reported separately from recommendation performance.

## 1. Configuration and imports

The stages here match `cartsense_analysis.analysis`: load and validate, preprocess, split users, build order baskets, mine rules, recommend, evaluate, and build the application payload.

In [1]:
from pathlib import Path
import json
import pandas as pd

from cartsense_analysis import (
    build_order_baskets,
    build_payload,
    evaluate_recommendations,
    load_orders,
    mine_association_rules,
    split_users,
)

DATA_PATH = Path('public/dataset/modeldataset.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('../public/dataset/modeldataset.csv')
SEED = 42
MIN_SUPPORT = 0.005
MIN_CONFIDENCE = 0.20
MAX_LEN = 3
KS = (1, 3, 5, 10)

## 2. Load, validate, and preprocess

The loader validates required identifiers and fields, keeps row-level errors, converts numeric columns, and preserves missing optional values as measurable quality information.

In [2]:
frame, quality = load_orders(DATA_PATH)
quality_summary = {
    'total_rows': quality['totalRows'],
    'valid_rows': quality['validRows'],
    'invalid_rows': quality['invalidRows'],
    'missing_values': quality['missingValues'],
}
pd.Series(quality_summary, dtype='object')

total_rows                                   999980
valid_rows                                   999980
invalid_rows                                      0
missing_values    {'days_since_prior_order': 61339}
dtype: object

In [3]:
required_columns = {'order_id', 'user_id', 'product_id', 'product_name', 'department'}
assert required_columns.issubset(frame.columns)
assert frame[['order_id', 'user_id', 'product_id']].notna().all().all()
assert frame['order_dow'].dropna().between(0, 6).all()
assert frame['order_hour_of_day'].dropna().between(0, 23).all()
frame.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,department_id,department,product_name
0,1201011,147243,14,0,16,3.0,94,1,0,7,beverages,tea
1,1201011,147243,14,0,16,3.0,83,2,0,4,produce,fresh vegetables
2,1201011,147243,14,0,16,3.0,83,3,1,4,produce,fresh vegetables
3,1201011,147243,14,0,16,3.0,24,4,1,4,produce,fresh fruits
4,1201011,147243,14,0,16,3.0,120,5,1,16,dairy eggs,yogurt


## 3. Order-level baskets and leakage-safe split

A basket is one order, so products from different departments can be associated. Users are split before mining and no user is present in both partitions.

In [4]:
train, test = split_users(frame, seed=SEED)
train_baskets = build_order_baskets(train)
test_baskets = build_order_baskets(test)
assert set(train['user_id']).isdisjoint(set(test['user_id']))
pd.DataFrame({
    'partition': ['train', 'test'],
    'users': [train['user_id'].nunique(), test['user_id'].nunique()],
    'orders': [len(train_baskets), len(test_baskets)],
    'rows': [len(train), len(test)],
})

,partition,users,orders,rows
0,train,41797,79102,800262
1,test,10449,19825,199718


## 4. Mine Apriori rules

Rules are mined only from training users for evaluation. Confidence filters the rules; support and lift provide reach and relative-association context.

In [5]:
rules, trace = mine_association_rules(
    train,
    min_support=MIN_SUPPORT,
    min_confidence=MIN_CONFIDENCE,
    max_len=MAX_LEN,
)
rule_metrics = {
    'baskets': trace['basketCount'],
    'frequent_itemsets': trace['itemsetCount'],
    'rules': len(rules),
    'min_support': MIN_SUPPORT,
    'min_confidence': MIN_CONFIDENCE,
}
pd.Series(rule_metrics)

baskets              79102.000
frequent_itemsets     5687.000
rules                 8791.000
min_support              0.005
min_confidence           0.200
dtype: float64

In [6]:
rules_frame = pd.DataFrame(rules)
rules_frame[['antecedentProductName', 'consequentProductName', 'support', 'confidence', 'lift', 'leverage', 'orders']].head(20)

,antecedentProductName,consequentProductName,support,confidence,lift,leverage,orders
0,frozen vegan vegetarian,tofu meat alternatives,0.005057,0.201005,5.845551,0.004192,400
1,"pasta sauce, bread",dry pasta,0.006763,0.398065,5.563211,0.005548,535
2,"bread, dry pasta",pasta sauce,0.006763,0.345384,5.490469,0.005532,535
3,"chips pretzels, dry pasta",pasta sauce,0.005904,0.337672,5.367868,0.004804,467
4,"canned meals beans, soup broth bouillon",canned jarred vegetables,0.005385,0.376325,5.231646,0.004356,426
5,"packaged cheese, dry pasta",pasta sauce,0.009873,0.328842,5.227506,0.007985,781
6,"pasta sauce, chips pretzels",dry pasta,0.005904,0.363707,5.083033,0.004742,467
7,"pasta sauce, packaged cheese",dry pasta,0.009873,0.356947,4.988555,0.007894,781
8,"pasta sauce, milk",dry pasta,0.006978,0.351816,4.916852,0.005559,552
9,"packaged vegetables fruits, dry pasta",pasta sauce,0.011150,0.300102,4.770634,0.008813,882


## 5. Recommendation evaluation

One product is hidden from every eligible test order. Recommendations are ranked by the strongest rule confidence; popularity is used as a fallback and as a baseline. The target is strong, reproducible recommendation quality, not an invented classification accuracy percentage.

In [7]:
evaluation = evaluate_recommendations(train, test, rules, ks=KS, seed=SEED)
evaluation_table = pd.DataFrame({
    'Popularity baseline': evaluation['popularityBaseline'],
    'Rules only': evaluation['rules'],
    'Rules + popularity fallback': evaluation['rulesWithPopularityFallback'],
}).T
evaluation_table

,coveragePercent,hitRateAt1,recallAt1,mrrAt1,hitRateAt3,recallAt3,mrrAt3,hitRateAt5,recallAt5,mrrAt5,hitRateAt10,recallAt10,mrrAt10
Popularity baseline,100.00,9.00,9.00,0.0900,20.39,20.39,0.1385,26.88,26.88,0.1533,40.32,40.32,0.1712
Rules only,99.48,16.17,16.17,0.1617,27.02,27.02,0.2088,34.30,34.30,0.2254,47.30,47.30,0.2426
Rules + popularity fallback,100.00,16.21,16.21,0.1621,27.09,27.09,0.2094,34.38,34.38,0.2260,47.62,47.62,0.2434
